<a href="https://colab.research.google.com/github/Malik-Nauman-Rauf-207/control-systems-book/blob/main/CIMID_Chapter4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🐾 Chapter 4: Vector Tames the Machine Lab

Welcome to Vector's Hardware Lab!

In Chapter 4, we learned that a real physical Arduino circuit is the best way to test our ideas. While you can wire up physical circuits in your browser using **[Wokwi (wokwi.com)](https://wokwi.com/arduino)**, it is also incredibly useful to simulate the invisible electrical signals that the Arduino uses to talk to the real world.

In this notebook, we will visualize the exact digital waves, ADC staircases, and PWM pulses that flow through the Arduino's tiny copper nerves!

### ⚙️ Step 0: Unlock the Signal Analyzers
**Click the "Play" [▶] button on the code cell below first!** It will load all our Python math and graphing libraries (`numpy`, `matplotlib`, and `ipywidgets`) so you can use the interactive sliders.

In [ ]:
# --- 🛠️ WORKSHOP SETUP: RUN THIS CELL FIRST! 🛠️ ---
print("⏳ Loading Vector's Signal Analyzers...")

import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

print("✅ Workshop Ready! You can now run the Arduino signal simulators below.")

---
## 💡 Lab 1: The Blink Timeline (Digital Pins)

An Arduino Digital Pin only knows two words: `HIGH` (5 Volts) and `LOW` (0 Volts).

When you write the `Blink` sketch, you tell the Arduino to switch between these two voltages on a rigid stopwatch schedule.

**Run the cell below and adjust the `ON` and `OFF` delays in milliseconds to see the digital square wave the Arduino sends to its LED!**

In [ ]:
# --- 💡 LAB 1: DIGITAL BLINK WAVEFORM ---
def run_blink_sim(on_ms=500, off_ms=500):
    on_s = on_ms / 1000.0
    off_s = off_ms / 1000.0
    period = on_s + off_s

    t = np.linspace(0, 5.0, 1000)
    # Modulo period to find where we are in each cycle
    cycle_pos = np.mod(t, period)
    voltage = np.where(cycle_pos < on_s, 5.0, 0.0)

    fig, ax = plt.subplots(figsize=(8, 3.5))
    ax.step(t, voltage, where="post", color="#FF9A3C", linewidth=3)
    ax.fill_between(t, voltage, step="post", color="#FF9A3C", alpha=0.2)
    ax.set_ylim(-0.5, 6.0)
    ax.set_yticks([0, 5])
    ax.set_yticklabels(["LOW (0V)\nLED OFF", "HIGH (5V)\nLED ON"])
    ax.set_xlabel("Time (seconds)", fontweight="bold")
    ax.set_title(f"Digital Output Wave (Cycle Time = {period:.1f}s)", fontweight="bold")
    ax.grid(True, linestyle=":", alpha=0.6)
    for spine in ["top", "right"]:
        ax.spines[spine].set_visible(False)
    plt.tight_layout()
    plt.show()

widgets.interact(
    run_blink_sim,
    on_ms=widgets.IntSlider(min=100, max=1500, step=100, value=500, description='ON (ms):'),
    off_ms=widgets.IntSlider(min=100, max=1500, step=100, value=500, description='OFF (ms):')
);

---
## 🎛️ Lab 3: Turning a Smooth Knob into Numbers (The ADC)

The real world is smooth and continuous (Analog). But computers can only think in discrete, choppy integer steps (Digital).

The Arduino UNO uses a **10-bit ADC (Analog-to-Digital Converter)** to chop voltages between 0V and 5V into exactly $1,024$ steps (Codes `0` to `1023`).

**Run the cell below and drag the ADC Resolution slider.** Watch how a low-bit ADC creates a clunky staircase, while a 10-bit ADC hugs the smooth physical voltage almost perfectly!

In [ ]:
# --- 🎛️ LAB 3: ADC QUANTIZATION STAIRCASE ---
def run_adc_sim(bits=3):
    levels = 2 ** bits
    max_code = levels - 1

    t = np.linspace(0, 10, 500)
    # Smooth physical voltage between 0V and 5V
    v_true = 2.5 + 2.3 * np.sin(2 * np.pi * t / 10.0)

    # Quantized ADC code and reconstructed voltage
    adc_code = np.round((v_true / 5.0) * max_code)
    v_quantized = (adc_code / max_code) * 5.0

    fig, ax = plt.subplots(figsize=(8, 3.5))
    ax.plot(t, v_true, color="#0077CC", linewidth=2, linestyle="--", alpha=0.7, label="True Smooth Voltage (0-5V)")
    ax.step(t, v_quantized, where="mid", color="#CC2222", linewidth=2.5, label=f"{bits}-Bit ADC ({levels} steps: 0 to {max_code})")
    ax.set_xlabel("Time (seconds)", fontweight="bold")
    ax.set_ylabel("Voltage (V)", fontweight="bold")
    ax.set_title(f"ADC Quantization Staircase ({bits} Bits = {levels} Codes)", fontweight="bold")
    ax.set_ylim(-0.3, 5.5)
    ax.grid(True, linestyle=":", alpha=0.6)
    ax.legend(loc="upper right")
    for spine in ["top", "right"]:
        ax.spines[spine].set_visible(False)
    plt.tight_layout()
    plt.show()

widgets.interact(
    run_adc_sim,
    bits=widgets.IntSlider(min=2, max=10, step=1, value=3, description='ADC Bits:')
);

---
## 🌟 Lab 4: Faking Smooth Brightness with PWM

How does an Arduino dim an LED if its digital pins can only output `5V` (Maximum Brightness) or `0V` (Completely Dark)?

It uses **Pulse-Width Modulation (PWM)**! It switches the 5V pin on and off almost 500 times per second. Your brain averages out the high-speed flashes into a smooth, dim light.

**Run the cell below and slide the `analogWrite` command (from `0` to `255`).** Notice how the pulses never get taller—they just get *wider* (increasing the Duty Cycle!).

In [ ]:
# --- 🌟 LAB 4: PWM DUTY CYCLE ---
def run_pwm_sim(cmd=128):
    duty = cmd / 255.0
    avg_v = 5.0 * duty

    t = np.linspace(0, 4.0, 1000) # 4 PWM cycles
    cycle_pos = np.mod(t, 1.0)
    wave = np.where(cycle_pos < duty, 5.0, 0.0)

    fig, ax = plt.subplots(figsize=(8, 3.5))
    ax.step(t, wave, where="post", color="#0077CC", linewidth=2.5, label="Fast Switching Pin 9 (0V or 5V)")
    ax.fill_between(t, wave, step="post", color="#0077CC", alpha=0.15)
    ax.axhline(avg_v, color="#FF9A3C", linestyle="--", linewidth=3, label=f"Average Effect = {avg_v:.2f}V ({duty*100:.0f}% Duty Cycle)")

    ax.set_ylim(-0.5, 5.8)
    ax.set_yticks([0, 2.5, 5])
    ax.set_xlabel("PWM Cycles (Happens ~500 times per second!)", fontweight="bold")
    ax.set_ylabel("Pin Voltage (V)", fontweight="bold")
    ax.set_title(f"analogWrite(9, {cmd}) -> {duty*100:.0f}% Duty Cycle", fontweight="bold")
    ax.grid(True, linestyle=":", alpha=0.6)
    ax.legend(loc="upper right")
    for spine in ["top", "right"]:
        ax.spines[spine].set_visible(False)
    plt.tight_layout()
    plt.show()

widgets.interact(
    run_pwm_sim,
    cmd=widgets.IntSlider(min=0, max=255, step=5, value=128, description='analogWrite:')
);

---
## 🧠 Lab 5: Closing the Loop (Hysteresis Memory Band)

When building an automatic night-light, noisy sensor readings can make a simple controller "chatter" (flickering the LED rapidly on and off when the reading sits right on the edge of the threshold).

By using **Hysteresis** (two thresholds separated by a memory band), the controller remembers what it was doing and ignores small wiggles!

**Run the cell below and increase the Sensor Noise.** Compare the chattering red line (Single Threshold) to the calm, stable purple line (Hysteresis Memory Band)!

In [ ]:
# --- 🧠 LAB 5: SINGLE THRESHOLD VS. HYSTERESIS ---
def run_hysteresis_sim(noise=25, width=150):
    rng = np.random.default_rng(4)
    time = np.arange(0, 20, 0.05)
    center = 525
    low = center - width / 2
    high = center + width / 2

    # Simulated noisy LDR darkness reading
    reading = np.clip(
        center + 110 * np.sin(2 * np.pi * time / 10)
        + rng.normal(0, noise, time.size), 0, 1023
    )

    # 1. Single threshold decision (chatter-prone!)
    single = (reading > center).astype(int)

    # 2. Hysteresis band decision (with memory!)
    remembered = []
    lamp_on = False
    for value in reading:
        if not lamp_on and value > high:
            lamp_on = True
        elif lamp_on and value < low:
            lamp_on = False
        remembered.append(int(lamp_on))
    remembered = np.array(remembered)

    count_single = np.count_nonzero(np.diff(single))
    count_band = np.count_nonzero(np.diff(remembered))

    fig, axes = plt.subplots(3, 1, figsize=(8, 6), sharex=True)
    axes[0].plot(time, reading, color="#0077CC", label="LDR Darkness Reading")
    axes[0].axhline(low, color="green", linestyle="--", label=f"BRIGHT_OFF ({int(low)})")
    axes[0].axhline(high, color="red", linestyle="--", label=f"DARK_ON ({int(high)})")
    axes[0].set_ylabel("ADC Code")
    axes[0].set_title("Noisy Light Sensor Reading (Higher = Darker)", fontweight="bold")
    axes[0].legend(loc="upper right", fontsize=8)

    axes[1].step(time, single, where="post", color="#CC2222")
    axes[1].set_ylabel("Lamp ON/OFF")
    axes[1].set_title(f"Single Threshold (525): {count_single} switches (Chattering!)", fontweight="bold")

    axes[2].step(time, remembered, where="post", color="#8E24AA", linewidth=2)
    axes[2].set_ylabel("Lamp ON/OFF")
    axes[2].set_xlabel("Time (seconds)", fontweight="bold")
    axes[2].set_title(f"Hysteresis Memory Band: Only {count_band} clean switches!", fontweight="bold")

    for axis in axes:
        axis.grid(True, linestyle=":", alpha=0.5)
    axes[1].set_yticks([0, 1])
    axes[2].set_yticks([0, 1])
    fig.tight_layout()
    plt.show()

widgets.interact(
    run_hysteresis_sim,
    noise=widgets.IntSlider(min=0, max=80, step=5, value=25, description='Sensor Noise:'),
    width=widgets.IntSlider(min=0, max=250, step=10, value=150, description='Band Width:')
);

---
## 🐾 Vector’s Brain Teasers (Interactive Scratchpad)

Try predicting the answer to each Brain Teaser from the book *before* testing the math in the **Python Scratchpad** below!

1. **A Slower Blink**
   Vector changes both `delay(500)` instructions in Lab 1 to `delay(1000)`.
   a. How many seconds is the LED on during each blink?
   b. How long does one complete ON + OFF cycle take?
   c. How many complete cycles happen in 10 seconds?
   d. Does changing the delay turn `Blink` into a closed-loop feedback controller?

2. **Find the Missing Path**
   An LED’s longer leg (anode) connects to Pin `9` through a $1\text{ k}\Omega$ resistor, but its shorter leg (cathode) is plugged into an empty row on the breadboard with nothing else connected.
   a. What connection is missing?
   b. Why won't electricity flow if there is only one wire going *to* the LED?
   c. What should you unplug *before* moving the wire?

3. **The Protective Resistor (Ohm's Law)**
   Use the approximate values: $5\text{ V}$ pin output, $2\text{ V}$ drop across the red LED, and a $1,000\text{ }\Omega$ resistor.
   a. How many volts remain across the resistor?
   b. Using $I = V / R$, what is the current in Amps and milliamps (mA)?
   c. Why should you *never* remove the resistor to make the LED brighter?

4. **Pressed Means Low**
   Our button in Lab 2 uses `pinMode(2, INPUT_PULLUP)` and connects Pin `2` to `GND` when pressed.
   a. What does `digitalRead(2)` return when the button is **not** pressed?
   b. What does `digitalRead(2)` return when the button **is** pressed?
   c. Why do we need a pull-up resistor instead of leaving an unpressed pin disconnected ("floating")?

5. **Count the ADC Codes**
   The Arduino UNO's 10-bit ADC produces integer codes from `0` to `1023`.
   a. How many total possible integer codes exist in that range? (Hint: count `0`!)
   b. Approximately what voltage on `A0` produces a reading of `512`?
   c. Does a reading of `512` automatically mean $51.2^\circ\text{C}$? Why or why not?

6. **A Halfway PWM Command**
   The knob reading on `A0` is `512`, and the program calculates `pwmCommand = reading / 4`.
   a. What number is sent to `analogWrite(9, pwmCommand)`?
   b. What is the approximate duty cycle percentage?
   c. Does Pin `9` output a steady $2.5\text{ V}$ DC voltage, or does it switch rapidly between `0V` and `5V`?

7. **The Memory Band (Hysteresis)**
   In Lab 5, we set `BRIGHT_OFF = 450` and `DARK_ON = 600`.
   a. The lamp is currently **OFF** and the darkness reading is `650`. What happens?
   b. The lamp is currently **ON** and the reading drops to `520`. What happens?
   c. The lamp is currently **OFF** and the reading is `520`. What happens?
   d. Why can the exact same sensor reading (`520`) produce two different lamp states without the controller being broken?

8. **Walk Through the Controller**
   The lamp starts **OFF** (`0`), with `BRIGHT_OFF = 450` and `DARK_ON = 600`. The sensor readings arrive in this order:
   `650, 580, 470, 430, 510, 620`
   Write down whether the lamp is ON (`1`) or OFF (`0`) after each reading!

9. **Open Loop or Closed Loop?**
   Classify each circuit below as **Open-Loop**, **Manual Command**, or **Closed-Loop Feedback**:
   a. Lab 1 (`Blink` running on a fixed 500 ms timer).
   b. Lab 4 (A human twists a potentiometer knob to dim an LED).
   c. Lab 5 (An LDR measures darkness, turns on an LED pointed at the LDR, and uses the new light reading to decide what to do next).

10. **The Pump Can Wait!**
    Vector wants to unplug the LED from Pin `9` and wire a greenhouse water pump directly into Pin `9` and `GND`.
    a. Why can't an Arduino signal pin power a heavy motor or water pump directly?
    b. Why is mixing water and an open breadboard dangerous?

In [ ]:
# --- 🐍 PYTHON SCRATCHPAD FOR VECTOR'S BRAIN TEASERS ---

# Brain Teaser 3: Ohm's Law
voltage_left = 5.0 - 2.0  # 5V pin minus 2V LED drop
resistor = 1000.0         # 1k Ohm
current_amps = voltage_left / resistor
print(f"BT 3 -> Current: {current_amps} Amps ({current_amps * 1000} mA)")

# Brain Teaser 5: ADC Resolution
steps = 1024
max_volts = 5.0
volts_per_step = max_volts / steps
print(f"BT 5 -> Each ADC step is worth exactly: {volts_per_step:.4f} Volts")

# Brain Teaser 6: PWM Math
reading = 512
pwmCommand = reading // 4  # Integer division
duty_cycle = (pwmCommand / 255.0) * 100
print(f"BT 6 -> PWM Command: {pwmCommand} | Duty Cycle: {duty_cycle:.1f}%")

# Brain Teaser 8: Hysteresis Memory Array Test
readings = [650, 580, 470, 430, 510, 620]
bright_off = 450
dark_on = 600

lamp_on = False
states = []

for r in readings:
    if not lamp_on and r > dark_on:
        lamp_on = True
    elif lamp_on and r < bright_off:
        lamp_on = False
    states.append(1 if lamp_on else 0)

print(f"BT 8 -> Sensor Readings: {readings}")
print(f"        Lamp States:     {states}")

# --- 🛠️ ADD YOUR OWN BRAIN TEASER MATH BELOW! 🛠️ ---